# Diabetes Hospital Readmission and Diabetes Medications

## Question
Among hospitalized patients with diabetes, is 30-day readmission related to the diabetes medications they received, and to whether those medications were changed during the stay?

## Data
- **Source:** [Diabetes 130-US Hospitals for Years 1999-2008, UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008)
- 101,766 hospital encounters, one row per encounter, with 23 medication columns.
- **Outcome used here:** `readmitted == '<30'` (readmitted within 30 days) versus all other encounters.
- Medication status values: `No` (not prescribed), `Steady`, `Up` (dose increased), `Down` (dose decreased).

## How to read the results
This analysis describes **associations**. A higher readmission rate among patients on a drug does not mean the drug caused it. Sicker patients are more likely to receive insulin or have doses changed.

## 1. Setup and Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
os.makedirs('figures', exist_ok=True)

df = pd.read_csv('data/diabetic_data.csv')
print('Rows, columns:', df.shape)
df[['readmitted', 'change', 'diabetesMed', 'insulin', 'metformin']].head()

## 2. Define the Outcome
`readmitted` has three values: `<30` (within 30 days), `>30` (after 30 days) and `NO` (not readmitted). Readmission within 30 days is the outcome hospitals are most often measured on, so it is coded as 1 and everything else as 0.

In [ ]:
print(df['readmitted'].value_counts())

df['readmit_30'] = (df['readmitted'] == '<30').astype(int)
overall_rate = df['readmit_30'].mean() * 100
print(f"\nOverall 30-day readmission rate: {overall_rate:.1f}%")

## 3. How Common Is Each Medication?
### What This Shows
The percent of encounters in which each diabetes medication was prescribed (any status other than `No`).
### How to Read This Chart
Longer bars mean more patients received that drug. Drugs given to very few patients cannot support reliable comparisons, so they are excluded from later charts.

In [ ]:
medication_columns = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                      'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                      'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                      'miglitol', 'troglitazone', 'tolazamide', 'examide',
                      'citoglipton', 'insulin', 'glyburide-metformin',
                      'glipizide-metformin', 'glimepiride-pioglitazone',
                      'metformin-rosiglitazone', 'metformin-pioglitazone']

use = (df[medication_columns] != 'No').mean().mul(100).sort_values()
fig, ax = plt.subplots(figsize=(9, 8))
use.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Percent of Encounters Where Each Medication Was Prescribed', fontsize=14, fontweight='bold')
ax.set_xlabel('Encounters (%)')
plt.tight_layout()
plt.savefig('figures/medication_usage.png', dpi=300, bbox_inches='tight')
plt.show()

single = [c for c in medication_columns if df[c].nunique() == 1]
print('Medications with only one category (not usable for comparison):', single)

## 4. Readmission Rate by Medication and Status
### What This Shows
For every medication and status (`Steady`, `Up`, `Down`, `No`), the percent of encounters ending in 30-day readmission, with the number of encounters in each group. Groups with fewer than 100 encounters are hidden because their rates are unstable.
### How to Read This Table
Compare each rate with the overall rate printed above. Rates well above it mean readmission was more common in that group.

In [ ]:
long = df.melt(id_vars='readmit_30', value_vars=medication_columns,
               var_name='medication', value_name='status')
rates = (long.groupby(['medication', 'status'])['readmit_30']
             .agg(rate_pct=lambda s: s.mean() * 100, encounters='count')
             .reset_index())
rates = rates[rates['encounters'] >= 100].round({'rate_pct': 2})

table_rates = rates.pivot(index='medication', columns='status', values='rate_pct')
table_rates['overall'] = round(overall_rate, 2)
table_rates

## 5. Insulin and Metformin in Detail
### What This Shows
Insulin and metformin are the two most widely used drugs in the data. The bars show the 30-day readmission rate for each status, and the dashed line is the overall rate.
### How to Read This Chart
A bar above the dashed line means a higher-than-average readmission rate for that group. Compare `Up` and `Down` (dose changes) with `Steady` and `No`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
order = ['No', 'Down', 'Steady', 'Up']
for ax, med in zip(axes, ['insulin', 'metformin']):
    r = (df.groupby(med)['readmit_30'].mean().mul(100).reindex(order))
    n = df[med].value_counts().reindex(order)
    r.plot(kind='bar', ax=ax, color='steelblue')
    ax.axhline(overall_rate, color='red', linestyle='--', label=f'Overall {overall_rate:.1f}%')
    ax.set_title(med.capitalize(), fontsize=13, fontweight='bold')
    ax.set_xlabel('Medication status')
    ax.set_ylabel('30-day readmission rate (%)')
    ax.set_xticklabels([f"{o}\n(n={n[o]:,})" for o in order], rotation=0)
    ax.legend()
plt.suptitle('30-Day Readmission Rate by Medication Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/insulin_metformin_readmission.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. On a Medication versus Not on It, for Every Drug
### What This Shows
For each medication given to at least 1,000 patients (and withheld from at least 1,000), the difference in 30-day readmission rate between patients who received it (any status other than `No`) and those who did not, in percentage points.
### How to Read This Chart
Bars to the right mean patients on the drug were readmitted more often; bars to the left mean less often. The labels give the number of patients on the drug. Remember this is an association and is not adjusted for how sick patients were.

In [ ]:
rows = []
for med in medication_columns:
    on = df[df[med] != 'No']
    off = df[df[med] == 'No']
    if len(on) >= 1000 and len(off) >= 1000:
        rows.append({'medication': med,
                     'on_drug_n': len(on),
                     'diff_pp': on['readmit_30'].mean() * 100 - off['readmit_30'].mean() * 100})
diff = pd.DataFrame(rows).sort_values('diff_pp')

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['firebrick' if v > 0 else 'steelblue' for v in diff['diff_pp']]
ax.barh(diff['medication'], diff['diff_pp'], color=colors)
for i, (v, n) in enumerate(zip(diff['diff_pp'], diff['on_drug_n'])):
    ax.text(v, i, f"  n={n:,}", va='center', ha='left' if v >= 0 else 'right', fontsize=8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Readmission Rate Difference: On Drug vs Not On Drug', fontsize=14, fontweight='bold')
ax.set_xlabel('Difference in 30-day readmission rate (percentage points)')
plt.tight_layout()
plt.savefig('figures/readmission_difference_by_drug.png', dpi=300, bbox_inches='tight')
plt.show()
diff.sort_values('diff_pp', ascending=False).round(2)

## 7. Medication Changes and Any Diabetes Medication
### What This Shows
Readmission rates for encounters where diabetes medications were changed during the stay (`change`) and where any diabetes medication was prescribed (`diabetesMed`).
### How to Read This Chart
Compare the two bars in each panel. `Ch` means a change was made; `No` means none was made.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, col, title in zip(axes, ['change', 'diabetesMed'],
                          ['Medication changed during stay', 'Any diabetes medication prescribed']):
    r = df.groupby(col)['readmit_30'].mean().mul(100)
    n = df[col].value_counts()
    r.plot(kind='bar', ax=ax, color='steelblue')
    ax.axhline(overall_rate, color='red', linestyle='--', label=f'Overall {overall_rate:.1f}%')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('30-day readmission rate (%)')
    ax.set_xticklabels([f"{i}\n(n={n[i]:,})" for i in r.index], rotation=0)
    ax.legend()
plt.tight_layout()
plt.savefig('figures/readmission_by_change.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Are the Differences Statistically Significant?
### What This Shows
A chi-square test asks whether a gap in readmission rates between groups is larger than chance alone would produce. A p-value below 0.05 is the usual cutoff for calling a difference statistically significant.
### How to Read This Output
Small p-values (printed as `< 0.001`) mean the difference is very unlikely to be random. A significant result does not show that the medication caused the difference, and a very large sample can make even tiny gaps significant, so read it together with the size of the gap in percentage points.

In [ ]:
from scipy.stats import chi2_contingency

def p_value(group):
    table = pd.crosstab(group, df['readmit_30'])
    return chi2_contingency(table)[1]

def fmt(p):
    return '< 0.001' if p < 0.001 else f'{p:.3f}'

tests = {
    'Insulin: on vs not on':                    df['insulin'] != 'No',
    'Insulin: status (No / Down / Steady / Up)': df['insulin'],
    'Metformin: on vs not on':                  df['metformin'] != 'No',
    'Metformin: status (No / Down / Steady / Up)': df['metformin'],
    'Any medication change during stay':        df['change'],
}
results = pd.DataFrame({'comparison': list(tests),
                        'p_value': [fmt(p_value(g)) for g in tests.values()]})
results

## 9. Key Numbers
The cell below prints the main figures from this analysis so the conclusions can be written from the actual results.

In [ ]:
top = diff.sort_values('diff_pp', ascending=False).iloc[0]
bot = diff.sort_values('diff_pp').iloc[0]
chg = df.groupby('change')['readmit_30'].mean().mul(100)
print(f"Overall 30-day readmission rate: {overall_rate:.1f}% of {len(df):,} encounters")
print(f"Largest positive difference: {top['medication']} (+{top['diff_pp']:.1f} points, n={int(top['on_drug_n']):,})")
print(f"Largest negative difference: {bot['medication']} ({bot['diff_pp']:.1f} points, n={int(bot['on_drug_n']):,})")
print("Readmission rate by medication change:")
print(chg.round(1).to_string())

## 10. Conclusions and Limitations

**Conclusions**
- Overall, 11.2% of the 101,766 encounters ended in readmission within 30 days.
- Insulin was associated with higher readmission. Patients who received insulin were readmitted about 2 percentage points more often than those who did not. The rate was highest when the insulin dose was changed (about 13 to 14% for `Down` and `Up`, compared with about 10% for `No` and 11% for `Steady`).
- Metformin was associated with lower readmission, about 2 percentage points below patients not on it, and the lowest rates were in the `Steady` and `Up` groups.
- Encounters in which any diabetes medication was changed had a higher readmission rate (11.8%) than encounters with no change (10.6%).
- The chi-square tests in section 8 report whether each of these gaps is larger than chance.

**Limitations**
- These are unadjusted comparisons. Age, diagnoses, length of stay and illness severity are not controlled for.
- Associations are not causal. Sicker patients are more likely to receive insulin or have doses adjusted, which can raise readmission rates regardless of the drug.
- Data covers 130 US hospitals from 1999 to 2008, and prescribing practice has changed since then.
- Several drugs were given to very few patients, so they are excluded from the comparison chart.

**Possible next steps:** logistic regression adjusting for age, prior inpatient visits and diagnoses, to test whether the medication associations hold after accounting for patient differences.